In [10]:
import simpy
import random
from parameters import DEFAULT_PARAMETERS

# Load the default scenario parameters for this notebook run.
params = DEFAULT_PARAMETERS

# Keep local names for the existing notebook code so the rest of the cells stay unchanged.
num_of_households = params.num_households
sms_response_probability = params.sms_response_probability
SEED = params.seed
rng = random.Random(SEED)
sms_timing = params.sms_timing  # day when SMS is sent for send_sms_on_day_n function
intervention_costs = params.intervention_costs

In [11]:
class Household:
    def __init__(self, id, attribute1, attribute2):
        self.id = id
        self.attribute1 = attribute1
        self.attribute2 = attribute2
        # Add more attributes as needed

        self.responded = rng.choice([True, False])  # this would be based on the attributes in the future, but for now it's random

#hcange how the household is represented when printed
    def __repr__(self):
        return (
            f"Household(id={self.id}, attribute1={self.attribute1}, "
            f"attribute2={self.attribute2}, responded={self.responded})"
        )

In [12]:
def create_cost_tracker():
    return {
        "total_cost": 0.0,
        "intervention_counts": {},
        "intervention_cost_totals": {},
    }


def record_intervention_cost(cost_tracker, intervention_type, units=1):
    unit_cost = intervention_costs.get(intervention_type, 0.0)
    total_cost = unit_cost * units

    cost_tracker["intervention_counts"][intervention_type] = (
        cost_tracker["intervention_counts"].get(intervention_type, 0) + units
    )
    cost_tracker["intervention_cost_totals"][intervention_type] = (
        cost_tracker["intervention_cost_totals"].get(intervention_type, 0.0) + total_cost
    )
    cost_tracker["total_cost"] += total_cost


def summarise_costs(cost_tracker):
    return {
        "total_cost": round(cost_tracker["total_cost"], 2),
        "intervention_counts": cost_tracker["intervention_counts"],
        "intervention_cost_totals": {
            intervention_type: round(total_cost, 2)
            for intervention_type, total_cost in cost_tracker["intervention_cost_totals"].items()
        },
    }


def send_sms_on_day_n(env, households, cost_tracker):
    yield env.timeout(sms_timing)

    print(f"Day {env.now}: sending SMS reminders")
    for household in households:
        if household.responded is False:
            print(f"SMS sent to household {household.id} on day {env.now}")
            record_intervention_cost(cost_tracker, "sms")
            if rng.random() < sms_response_probability:
                household.responded = True
                print(f"Household {household.id} responded after SMS")


In [13]:
def send_sms_every_day(env, households, cost_tracker):
    while True:
        yield env.timeout(1)  # wait for 1 day
        for household in households:
            if household.responded is False:
                print(f"SMS sent to household {household.id} on day {env.now}")
                record_intervention_cost(cost_tracker, "sms")
                if rng.random() < sms_response_probability:  # again, this would be based on the attributes in the future, but for now it's a flat measure
                    household.responded = True
                    print(f"Household {household.id} responded after SMS")

In [14]:
def daily_responce_totals(env, households):
    response_list = []
    while True:
        yield env.timeout(1)
        response_list.append(sum(h.responded for h in households))
        print(f"on day {env.now}, the last 3 days of response are {response_list[-3:]}")


In [ ]:
#setup the environment
env = simpy.Environment()
cost_tracker = create_cost_tracker()


#generate households, attributes range from 1 to 10
hh_list = [Household(id=i, attribute1=rng.randint(1, 10), attribute2=rng.randint(1, 10)) for i in range(num_of_households)]
for h in hh_list:
    print(h)

#processes, #out those that are not needed
##env.process(send_sms_on_day_n(env, hh_list, cost_tracker))
env.process(send_sms_every_day(env, hh_list, cost_tracker))

#run simulation
env.run(until = 100)

#print results
for i in hh_list:
    print(f"Household {i.id} responded: {i.responded}")

cost_summary = summarise_costs(cost_tracker)
print(f"Total intervention cost: {cost_summary['total_cost']:.2f}")
print(f"Intervention counts: {cost_summary['intervention_counts']}")
print(f"Intervention cost totals: {cost_summary['intervention_cost_totals']}")

In [ ]:
#add results to a results list for later use
results_list = []
cost_summary = summarise_costs(cost_tracker)
cost_summary["scenario_name"] = params.scenario_name  # Add the scenario name to the cost summary
results_list.append(cost_summary)
print (results_list)

AttributeError: 'dict' object has no attribute 'append'